## Week 2 Day 2

¡¡Nuestro primer proyecto de Agentic Framework!!

Prepárate para algo ridículamente fácil.

Vamos a construir un sistema de Agente simple para generar correos electrónicos de alcance de ventas en frío:
1. Flujo de trabajo del Agente
2. Uso de herramientas para llamar a las funciones
3. Colaboración de los agentes mediante herramientas y traspasos

## Before we start - some setup:


Ppor favor visite Sendgrid en: https://sendgrid.com/

(Sendgrid es una empresa de Twilio para el envío de correos electrónicos).

Si SendGrid le da problemas, vea la implementación alternativa usando "Resend Email" en community_contributions/2_lab2_with_resend_email

Por favor, crea una cuenta - ¡es gratis! (al menos, para mí, ahora mismo).

Una vez que haya creado una cuenta, haga clic en:

Configuración (barra lateral izquierda) >> Claves API >> Crear clave API (botón arriba a la derecha)

Copia la clave en el portapapeles y añade una nueva línea a tu archivo .env:


`SENDGRID_API_KEY=xxxx`

Y también, dentro de SendGrid, vaya a:

Configuración (barra lateral izquierda) >> Autenticación del remitente >> "Verificar un único remitente"  
y verifique que su propia dirección de correo electrónico es una dirección real, para que SendGrid pueda enviar correos electrónicos por usted.


In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio



In [2]:
load_dotenv(override=True)

True

In [3]:
# Let's just check emails are working for you

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("dnaranjo88@gmail.com")  # Change to your verified sender
    to_email = To("zews.kevina@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

202


In [4]:
# pip install sendgrid
import os
from sendgrid import SendGridAPIClient
from sendgrid.helpers.mail import Mail

# --- CONFIG ---
FROM_EMAIL = "dnaranjo88@gmail.com"  # <-- Debe estar VERIFICADO en SendGrid
TO_EMAIL = "dnaranjo88@gmail.com"
SUBJECT = "Prueba SendGrid"
BODY = "Prueba mínima de envío via SendGrid (diagnóstico 403)."

# Opción A: lee la API key del entorno
API_KEY = os.environ.get("SENDGRID_API_KEY")

# Opción B (solo para prueba rápida): pega temporalmente tu API key
# API_KEY = "SG.xxxxxx..."

def send_test():
    if not API_KEY or not API_KEY.startswith("SG."):
        raise RuntimeError("API key ausente o inválida. Define SENDGRID_API_KEY o asigna API_KEY.")

    message = Mail(
        from_email=FROM_EMAIL,
        to_emails=TO_EMAIL,
        subject=SUBJECT,
        plain_text_content=BODY,
    )

    try:
        sg = SendGridAPIClient(API_KEY)
        resp = sg.send(message)
        print("Status:", resp.status_code)      # 202 esperado si envía
        print("Headers:", dict(resp.headers))
        # body suele venir vacío si 202; si hay error 4xx, trae detalle JSON
        if resp.body:
            print("Body:", resp.body)
    except Exception as e:
        # El SDK envuelve la respuesta 4xx/5xx; intentamos extraer info útil
        print("ERROR:", repr(e))
        try:
            # algunos errores exponen e.body o e.args[0].body
            print("Error body:", e.body)  # puede no existir según versión
        except Exception:
            pass

if __name__ == "__main__":
    send_test()


Status: 202
Headers: {'Server': 'nginx', 'Date': 'Tue, 23 Sep 2025 22:44:12 GMT', 'Content-Length': '0', 'Connection': 'close', 'X-Message-Id': 'HfsM-A5QSRmYmbSs6tXoSw', 'Access-Control-Allow-Origin': 'https://sendgrid.api-docs.io', 'Access-Control-Allow-Methods': 'POST', 'Access-Control-Allow-Headers': 'Authorization, Content-Type, On-behalf-of, x-sg-elas-acl', 'Access-Control-Max-Age': '600', 'X-No-CORS-Reason': 'https://sendgrid.com/docs/Classroom/Basics/API/cors.html', 'Strict-Transport-Security': 'max-age=31536000; includeSubDomains', 'Content-Security-Policy': "frame-ancestors 'none'", 'Cache-Control': 'no-cache', 'X-Content-Type-Options': 'no-sniff', 'Referrer-Policy': 'strict-origin-when-cross-origin'}


### ¿Ha recibido el correo electrónico de prueba?

Si obtienes un 202, ya puedes irte.

#### Certificate error

If you get an error SSL: CERTIFICATE_VERIFY_FAILED then students Chris S and Oleksandr K have suggestions:  
First run this: `!uv pip install --upgrade certifi`  
Next, run this:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

#### Other errors or no email

If there are other problems, you'll need to check your API key and your verified sender email address in the SendGrid dashboard

Or use the alternative implementation using "Resend Email" in community_contributions/2_lab2_with_resend_email

(Or - you could always replace the email sending code below with a Pushover call, or something to simply write to a flat file)

## Step 1: Agent workflow

In [5]:
instructions1 = "Usted es un agente de ventas que trabaja para ComplAI, \
una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
Escribes correos electrónicos fríos profesionales y serios"

instructions2 = "Eres un agente de ventas humorístico y atractivo que trabaja para ComplAI, \
una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
Escribes correos electrónicos en frío ingeniosos y atractivos que probablemente obtengan respuesta."

instructions3 = "Usted es un ocupado agente de ventas que trabaja para ComplAI, \
una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. \
Escribes correos electrónicos en frío concisos y directos"

In [6]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini"
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini"
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini"
)

In [7]:

result = Runner.run_streamed(sales_agent1, input="Escribir un correo electrónico de venta en frío")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Asunto: Optimice su Cumplimiento de SOC2 con ComplAI

Estimado/a [Nombre del Destinatario],

Espero que este mensaje lo/a encuentre bien. Me presento, soy [Tu Nombre], [Tu Cargo] en ComplAI, una empresa especializada en soluciones de cumplimiento y auditoría.

Entendemos que el cumplimiento de SOC2 puede ser un proceso complejo y a menudo intimidante. Con la creciente importancia de la seguridad de datos y la protección del cliente, contar con una herramienta efectiva y eficiente es crucial para su organización.

ComplAI es una solución SaaS impulsada por inteligencia artificial que simplifica el proceso de cumplimiento de SOC2 y prepara a su empresa para auditorías, permitiéndole:

1. **Automatizar tareas repetitivas**: Reduzca el tiempo y los recursos dedicados al cumplimiento.
2. **Identificar y mitigar riesgos**: Mejore la seguridad general de su organización anticipando potenciales vulnerabilidades.
3. **Generar informes precisos**: Simplifique la documentación requerida para audi

In [8]:
message = "Escribir un correo electrónico de venta en frío"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Asunto: Optimice su Cumplimiento de SOC2 con ComplAI

Estimado [Nombre del destinatario],

Espero que este mensaje le encuentre bien.

Me gustaría presentarle a ComplAI, una solución SaaS innovadora diseñada para ayudar a empresas como la suya a garantizar el cumplimiento de SOC2 y facilitar el proceso de auditoría. Entendemos que mantener altos estándares de seguridad y privacidad es fundamental para su negocio, y nuestra herramienta impulsada por inteligencia artificial optimiza este proceso de manera eficiente.

Con ComplAI, podrá:

1. **Automatizar la recopilación de evidencia**: Reduzca drásticamente el tiempo dedicado a la preparación de auditorías.
2. **Evaluar riesgos en tiempo real**: Identifique vulnerabilidades antes de que se conviertan en problemas.
3. **Aumentar su confianza en el cumplimiento**: Mejore la transparencia y seguridad de sus procesos, fortaleciendo la confianza de sus clientes.

Me encantaría programar una breve llamada para discutir cómo ComplAI puede adapt

In [9]:
sales_picker = Agent(
    name="sales_picker",
    instructions="Elige el mejor correo electrónico de venta en frío entre las opciones dadas. \
                  Imagina que eres un cliente y elige el que más probabilidades tenga de responder. \
                  No des explicaciones; responde sólo con el correo electrónico seleccionado.",
    model="gpt-4o-mini"
)

In [10]:
message = "Escribir un correo electrónico de venta en frío"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
Asunto: ¡Prepárate para la auditoría sin sudar!

Hola [Nombre],

¿Recuerdas la última vez que preparaste una auditoría? Esa mezcla de sudor frío y café en exceso. 😅

En ComplAI, entendemos que la conformidad con SOC2 puede sentirse como un maratón de obstáculos, pero ¿y si te dijera que tenemos la varita mágica que te ayudará a sortearlo con gracia?

Nuestra herramienta SaaS impulsada por IA no solo te ayuda a garantizar el cumplimiento de SOC2, sino que también transforma el proceso en algo tan sencillo como un café (sin cafeína). Desde la organización de documentos hasta la preparación de auditorías, te tenemos cubierto.

Imagina tener un asistente virtual que te ayude a evitar ese caos de última hora y, lo mejor de todo, ¡te permita celebrar con una buena copa al finalizar!

¿Te gustaría ver cómo ComplAI puede convertir esas noches de insomnio en dulces sueños? Estaré encantado de programar una llamada para mostrarte nuestras soluciones.

Saludos (y un café virtual

Now go and check out the trace:

https://platform.openai.com/traces

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [11]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini",
)

In [12]:
sales_agent1

Agent(name='Professional Sales Agent', instructions='Usted es un agente de ventas que trabaja para ComplAI, una empresa que ofrece una herramienta SaaS para garantizar el cumplimiento de SOC2 y prepararse para auditorías, impulsada por IA. Escribes correos electrónicos fríos profesionales y serios', prompt=None, handoff_description=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, metadata=None, store=None, include_usage=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), tools=[], mcp_servers=[], mcp_config={}, input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [13]:
@function_tool
def send_email(body: str):
    """ Enviar un correo electrónico con el Body dado a todos los prospectos de ventas """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("dnaranjo88@gmail.com")  # Change to your verified sender
    to_email = To("zews.kevina@gmail.com")  # Change to your recipient
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

### This has automatically been converted into a tool, with the boilerplate json created

In [14]:
# Let's look at it
send_email

FunctionTool(name='send_email', description='Enviar un correo electrónico con el Body dado a todos los prospectos de ventas', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000227FC4672E0>, strict_json_schema=True, is_enabled=True)

### And you can also convert an Agent into a tool

In [15]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000227FC618400>, strict_json_schema=True, is_enabled=True)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [16]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000227FA286A20>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000227FC4B2B60>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'

## And now it's time for our Sales Manager - our planning agent

In [17]:
# Improved instructions thanks to student Guillermo F.

instructions = """
Eres Director de Ventas en ComplAI. Su objetivo es encontrar el mejor correo electrónico de ventas en frío utilizando las herramientas sales_agent.
 
Siga estos pasos cuidadosamente:
1. 1. Genere borradores: Utilice las tres herramientas sales_agent para generar tres borradores de correo electrónico diferentes. No continúe hasta que los tres borradores estén listos.
 
2. 2. Evaluar y seleccionar: Revise los borradores y elija el mejor correo electrónico utilizando su juicio sobre cuál es el más efectivo.
 
3. 3. Utilice la herramienta send_email para enviar el mejor mensaje (y sólo el mejor) al usuario.
 
Reglas cruciales:
- Debe utilizar las herramientas del agente de ventas para generar los borradores - no los escriba usted mismo.
- Debe enviar UN email usando la herramienta send_email - nunca más de uno.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model="gpt-4o-mini")

message = "Enviar un correo electrónico de ventas en frío dirigido a 'Estimado Director General'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Wait - you didn't get an email??</h2>
            <span style="color:#ff7800;">With much thanks to student Chris S. for describing his issue and fixes. 
            If you don't receive an email after running the prior cell, here are some things to check: <br/>
            First, check your Spam folder! Several students have missed that the emails arrived in Spam!<br/>Second, print(result) and see if you are receiving errors about SSL. 
            If you're receiving SSL errors, then please check out theses <a href="https://chatgpt.com/share/680620ec-3b30-8012-8c26-ca86693d0e3d">networking tips</a> and see the note in the next cell. Also look at the trace in OpenAI, and investigate on the SendGrid website, to hunt for clues. Let me know if I can help!
            </span>
        </td>
    </tr>
</table>

### And one more suggestion to send emails from student Oleksandr on Windows 11:

If you are getting certificate SSL errors, then:  
Run this in a terminal: `uv pip install --upgrade certifi`

Then run this code:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

Thank you Oleksandr!

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!!


### Los Handoffs representan una forma en que un agente puede delegar en otro, pasándole el control

Las Handoffs y los Agents-as-tools como herramientas son similares:

En ambos casos, un Agente puede colaborar con otro Agente

Con las herramientas, el control se devuelve

Con las transferencias, el control pasa de un Agente a otro


In [38]:

subject_instructions = "Puedes escribir un asunto para un correo electrónico de ventas en frío. \
Se le da un mensaje y tiene que escribir un asunto para un correo electrónico que es probable que obtenga una respuesta."

html_instructions = "Puede convertir un cuerpo de correo electrónico de texto en un cuerpo de correo electrónico HTML. \
Se le da un cuerpo de correo electrónico de texto que podría tener algunos markdown \
y tienes que convertirlo en un cuerpo de correo electrónico HTML con un diseño bonito, claro pero con titulos y colores. Ademas que sea convincente."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Escribir un asunto para un correo electrónico de venta en frío")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convertir un cuerpo de correo electrónico de texto en un cuerpo de correo electrónico HTML")


In [39]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Envíe un correo electrónico con el asunto y el cuerpo HTML indicados a todos los clientes potenciales. """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("dnaranjo88@gmail.com")  # Change to your verified sender
    to_email = To("zews.kevina@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [40]:
tools = [subject_tool, html_tool, send_html_email]

In [41]:
tools

[FunctionTool(name='subject_writer', description='Escribir un asunto para un correo electrónico de venta en frío', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000227FE133CE0>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='html_converter', description='Convertir un cuerpo de correo electrónico de texto en un cuerpo de correo electrónico HTML', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000227FE133880>, strict_json_schema=True, is_enabled=True),
 FunctionTool(name='send_html_email', desc

In [42]:
instructions ="Usted es un formateador y remitente de correo electrónico. Recibe el cuerpo de un correo electrónico para enviarlo. \
Primero utiliza la herramienta subject_writer para escribir un asunto para el correo electrónico, luego utiliza la herramienta html_converter para convertir el cuerpo a HTML. \
Por último, utilice la herramienta send_html_email para enviar el correo electrónico con el asunto y el cuerpo HTML."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model="gpt-4o-mini",
    handoff_description="Convertir un correo electrónico en HTML y enviarlo")


### Now we have 3 tools and 1 handoff

In [43]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000227FA286A20>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x00000227FC4B2B60>, strict_json_schema=True, is_enabled=True), FunctionTool(name='sales_agent3', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}

In [44]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
Eres Director de Ventas en ComplAI. Su objetivo es encontrar el mejor correo electrónico de ventas en frío utilizando las herramientas sales_agent.
 
Siga estos pasos cuidadosamente:
1. 1. Genere borradores: Utilice las tres herramientas sales_agent para generar tres borradores de correo electrónico diferentes. No continúe hasta que los tres borradores estén listos.
 
2. 2. Evaluar y seleccionar: Revise los borradores y elija el mejor correo electrónico según su criterio.
Puedes utilizar las herramientas varias veces si no estás satisfecho con los resultados del primer intento.
 
3. 3. Entrega para el envío: Pase ÚNICAMENTE el borrador del email ganador al agente 'Email Manager'. El Email Manager se encargará del formateo y del envío.
 
Reglas cruciales:
- Debe utilizar las herramientas del agente de ventas para generar los borradores - no los escriba usted mismo.
- Debe entregar exactamente UN email al Email Manager - nunca más de uno.

"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model="gpt-4o-mini")

message = "Envía un correo electrónico de ventas en frío dirigido a Dear CEO from Kevin"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">¿Puede identificar los patrones de diseño Agentic que se utilizaron aquí?<br/>
            ¿Cuál es la 1 línea que hizo que esto pasara de ser un "flujo de trabajo" de Agentic a un "agente" según la definición de Anthropic?<br/>
            Intente añadir más herramientas y agentes. Podría tener herramientas que gestionen la combinación de correspondencia para enviar a una lista.<br/><br/>
            DESAFÍO DIFÍCIL: investigue cómo puede hacer que SendGrid llame a un webhook Callback cuando un usuario responde a un correo electrónico,
            Luego hacer que el SDR responda para mantener la conversación. Esto puede requerir algo de "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

## Extra note:

Google ha lanzado su Kit de Desarrollo de Agentes (ADK). Todavía no ha conseguido la tracción de los otros frameworks de este curso, pero está llamando la atención. Es interesante observar que se parece bastante a OpenAI Agents SDK. Para darte una idea, aquí tienes un ejemplo de código de ADK:

```
root_agent = Agente(
    name="agente_tiempo_tiempo",
    model="gemini-2.0-flash",
    description="Agente para responder preguntas sobre la hora y el tiempo en una ciudad",
    instruction="Eres un agente útil que puede responder a las preguntas de los usuarios sobre la hora y el tiempo en una ciudad",
    tools=[get_weather, get_current_time]
)
```

Vaya, ¡me suena!

Y un estudiante ha contribuido con un agente de atención al cliente en community_contributions que utiliza ADK.

